In [ ]:
#| default_exp media

# Media

> Media files and the AnkiWeb media sync protocol

Media sync exchanges files with AnkiWeb independently of the notes and cards in collection sync. Neither protocol requires the other to run first. Media sync has its own update sequence numbers (USNs) and is always incremental.

fastanki keeps the files beside the collection and tracks their sync state in a sqlite database. Before syncing, it checks the folder for changes. The implementation follows Anki's Rust code in `rslib/src/sync/media/`. The examples below compare it with Anki as a second client.

In [ ]:
import zipfile, io, hashlib, unicodedata, json
from fastcore.utils import *
from fastanki.schema import *
from fastanki.collection import *
from fastanki.syncer import *

In [ ]:
import tempfile
from fastcore.test import *
from anki.collection import Collection as AnkiCollection
from anki.utils import checksum

## Filenames

Devices identify a media file by its filename. `norm_fname` follows Anki's rules for names that work across platforms:

- Convert to Unicode NFC.
- Strip disallowed characters.
- Add underscores to Windows device names and names ending in a dot or space.
- Limit the name to 120 UTF-8 bytes.

Sync skips local files whose names don't meet these rules. macOS needs special handling because it lists names in NFD form.

In [ ]:
MAX_FNAME_BYTES = 120
_WIN_DEVICE = re.compile(r'(?i)^(CON|PRN|AUX|NUL|COM[1-9]|LPT[1-9])(\.|$)')
_NONSYNCABLE = re.compile(r'(?i)^(thumbs\.db|\.ds_store)$')

def _disallowed(c): return c in '[]<>:"/?*^\\|' or c<='\x1f' or c=='\x7f' or unicodedata.category(c)=='Cn'

def _trunc_bytes(s, n):
    "Truncate `s` to at most `n` utf-8 bytes, on a character boundary"
    while len(s.encode())>n: s = s[:-1]
    return s

def _stem_ext(fname, max_bytes):
    "Split into (stem, ext), each truncated so the joined name fits `max_bytes` (ext capped at 10 bytes)"
    stem,dot,ext = fname.rpartition('.')
    if not dot: stem,ext = ext,''
    ext = _trunc_bytes(ext, 10)
    return _trunc_bytes(stem, max_bytes-len(ext.encode())-2), ext

def norm_fname(fname):
    "Normalize a media filename as Anki does: NFC, problem characters stripped, device names defused, 120 bytes max"
    fname = unicodedata.normalize('NFC', fname)
    fname = ''.join(c for c in fname if not _disallowed(c)).replace('\xa0', ' ')
    fname = _WIN_DEVICE.sub(r'\1_\2', fname)
    if len(fname.encode())>MAX_FNAME_BYTES:
        stem,ext = _stem_ext(fname, MAX_FNAME_BYTES)
        fname = f'{stem}.{ext}' if ext else stem
    if fname.endswith((' ', '.')): fname += '_'
    return fname

def _hash_name(fname, csum):
    "foo.jpg -> foo-{csum}.jpg, for renaming a file whose name is taken by different content"
    stem,ext = _stem_ext(fname, MAX_FNAME_BYTES-41)
    return f'{stem}-{csum}.{ext}'

Compare filename normalization with Anki:

In [ ]:
test_eq(norm_fname('foo.jpg'), 'foo.jpg')
test_eq(norm_fname('con.jpg[]><:"/?*^\\|\0\r\n'), 'con_.jpg')
test_eq(norm_fname('test.'), 'test._')
test_eq(norm_fname('test '), 'test _')
test_eq(norm_fname('x'*130+'.jpg'), 'x'*115+'.jpg')

td = Path(tempfile.mkdtemp())
ac = AnkiCollection(str(td/'oracle.anki2'))
names = ['ok.jpg', 'con.', 'foo bar.png', 'te\u0301st.gif', 'x'*130+'.webp']   # NFD input comes back NFC
for name in names: test_eq(ac.media.write_data(name, b'data'), norm_fname(name))
[(name, norm_fname(name)) for name in names[:4]]

[('ok.jpg', 'ok.jpg'),
 ('con.', 'con_._'),
 ('foo bar.png', 'foo bar.png'),
 ('tést.gif', 'tést.gif')]

## Adding files

`add_media_file` preserves existing files. If the name and content match, it returns the name without writing. If the content differs, it writes to `name-{sha1}.ext`.

Before adding the checksum, it retries names containing uppercase letters in lowercase. This also applies to new files: adding `Foo.jpg` creates `foo.jpg`.

In [ ]:
def _sha1(data): return hashlib.sha1(data).hexdigest()

def add_media_file(folder, data, fname):
    "Write `data` into `folder` under (normalized) `fname` without clobbering different content; returns the name used"
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    fname = norm_fname(fname)
    csum = _sha1(data)
    p = folder/fname
    if p.exists() and _sha1(p.read_bytes())==csum: return fname
    if fname.lower()!=fname: return add_media_file(folder, data, fname.lower())
    if p.exists(): fname = _hash_name(fname, csum)
    (folder/fname).write_bytes(data)
    return fname

On a case-insensitive filesystem, `Foo.jpg` matches an existing `foo.jpg`. If the content matches too, `add_media_file` returns `Foo.jpg` without changing its case. Compare these choices with Anki:

In [ ]:
mf = td/'add'
cases = [('Foo.jpg', b'aaa'), ('Foo.jpg', b'aaa'), ('Foo.jpg', b'bbb'), ('dup.png', b'x'), ('dup.png', b'y')]
theirs = [ac.media.write_data(n, d) for n,d in cases]
ours = [add_media_file(mf, d, n) for n,d in cases]
test_eq(ours, theirs)                              # we make exactly the oracle's choices, branch by branch
test_eq(_sha1(b'aaa'), checksum(b'aaa'))           # and our checksums are Anki's: hex sha1
test_eq(theirs[0], 'foo.jpg')                      # fresh uppercase is stored lowercased
test_eq(theirs[2], f'foo-{_sha1(b"bbb")}.jpg')     # same name, different content: hash-renamed
test_eq(theirs[3:], ['dup.png', f'dup-{_sha1(b"y")}.png'])
ours

['foo.jpg',
 'Foo.jpg',
 'foo-5cb138284d431abd6a053a56625ec088bfb88912.jpg',
 'dup.png',
 'dup-95cb0bfd2977c761298d9624e4b4d4c72a39974a.png']

## The media database

For `collection.anki2`, fastanki uses the folder `collection.media` and database `collection.mdb`, matching Anki's paths.

The `media` table tracks each file's checksum, mtime and upload status. A null `csum` records a deletion. `dirty` marks entries awaiting upload. The `meta` table stores the folder's last observed mtime and the last server USN processed.

In [ ]:
MEDIA_SCHEMA = r"""
CREATE TABLE media (
  fname text NOT NULL PRIMARY KEY,
  csum text,            -- NULL: deleted, deletion pending upload
  mtime int NOT NULL,   -- 0 if deleted
  dirty int NOT NULL
) without rowid;
CREATE INDEX idx_media_dirty ON media (dirty) WHERE dirty=1;
CREATE TABLE meta (dirMod int, lastUsn int);
INSERT INTO meta VALUES (0, 0);
"""

class Media:
    "A collection's media folder plus the db tracking its sync state"
    def __init__(self, folder, db_path):
        self.folder = Path(folder)
        self.folder.mkdir(parents=True, exist_ok=True)
        self.con = connect(db_path)
        if not self.con.execute("select 1 from sqlite_master where name='media'").fetchone(): self.con.execute(MEDIA_SCHEMA)
    def close(self): self.con.close()
    def __enter__(self): return self
    def __exit__(self, *args): self.close()
    def entry(self, fname):
        "(csum, mtime, dirty) for `fname`, or None"
        return self.con.execute('select csum, mtime, dirty from media where fname=?', (fname,)).fetchone()
    def set_entry(self, fname, csum, mtime, dirty):
        self.con.execute('insert or replace into media values (?,?,?,?)', (fname, csum, mtime, int(dirty)))
    def meta(self):
        "(dirMod, lastUsn)"
        return self.con.execute('select dirMod, lastUsn from meta').fetchone()
    def set_meta(self, dirmod, last_usn): self.con.execute('update meta set dirMod=?, lastUsn=?', (dirmod, last_usn))
    def count(self): return self.con.execute('select count(*) from media where csum is not null').fetchone()[0]
    def pending(self, limit):
        "Entries awaiting upload, as [(fname, csum, mtime)] with csum None for deletions"
        return self.con.execute('select fname, csum, mtime from media where dirty=1 limit ?', (limit,)).fetchall()
    def force_resync(self):
        "Clear all sync state, so the next sync re-lists everything against the server"
        self.con.execute('delete from media; update meta set lastUsn=0, dirMod=0')

In [ ]:
m = Media(td/'m'/'media', td/'m'/'collection.mdb')
test_eq(m.count(), 0)
test_is(m.entry('test.mp3'), None)
m.set_entry('test.mp3', None, 0, False)
test_eq(m.entry('test.mp3'), (None, 0, 0))
m.set_entry('test.mp3', _sha1(b'hello'), 123, True)
test_eq(m.pending(25), [('test.mp3', _sha1(b'hello'), 123)])
test_eq(m.count(), 1)
m.set_meta(123, 321)
test_eq(m.meta(), (123, 321))
m.force_resync()
test_eq((m.count(), m.meta()), (0, (0, 0)))

## Change tracking

The scan records additions, changes and deletions for upload. It hashes new files and files whose mtimes have changed. A changed mtime alone doesn't require an upload if the checksum still matches. Missing files get null checksums and await deletion uploads.

The scan skips the folder if its mtime matches `dirMod`, stored in milliseconds. It compares individual file mtimes in seconds, matching Anki's database format.

The scan excludes empty files, files over 100MiB, `thumbs.db`, `.ds_store` and invalid filenames. On macOS it accepts NFD filenames whose NFC equivalents are valid.

In [ ]:
MAX_FILE_SIZE = 100*1024*1024

def _mtime_ms(p): return int(p.stat().st_mtime*1000)
def _mtime_s(p): return int(p.stat().st_mtime)

def _syncable_fname(fname):
    "The NFC form of `fname` if sync can use it, else None"
    nfc = unicodedata.normalize('NFC', fname)
    if norm_fname(nfc)!=nfc or _NONSYNCABLE.match(nfc): return None
    if fname!=nfc and sys.platform!='darwin': return None    # only macOS serves NFD names for NFC paths
    return nfc

@patch
def _register_changes(self:Media):
    "Reconcile the db with the folder: new/changed files become dirty entries, missing files deletion tombstones"
    dirmod = _mtime_ms(self.folder)
    dirmod0,last_usn = self.meta()
    if dirmod==dirmod0: return
    self.con.execute('BEGIN IMMEDIATE')
    try:
        mtimes = dict(self.con.execute('select fname, mtime from media where csum is not null'))
        for p in self.folder.iterdir():
            fname = _syncable_fname(p.name)
            if fname is None or not p.is_file(): continue
            sz = p.stat().st_size
            if not sz or sz>MAX_FILE_SIZE: continue
            prev = mtimes.pop(fname, None)
            mtime = _mtime_s(p)
            if prev==mtime: continue
            csum,cur = _sha1(p.read_bytes()),self.entry(fname)
            dirty = cur[2] if prev is not None and cur and cur[0]==csum else True   # mtime bumped, content unchanged: keep flag
            self.set_entry(fname, csum, mtime, dirty)
        for fname in mtimes: self.set_entry(fname, None, 0, True)                   # in the db, gone from disk
        self.set_meta(dirmod, last_usn)
        self.con.execute('COMMIT')
    except: self.con.execute('ROLLBACK'); raise

Check additions, modifications and deletions. Backdating mtimes lets the test trigger scans without waiting for the clock:

In [ ]:
def _backdate(p):
    t = p.stat().st_mtime - 3
    os.utime(p, (t, t))

trk = Media(td/'trk'/'media', td/'trk'/'collection.mdb')
f1 = trk.folder/'file.jpg'
f1.write_bytes(b'hello')
trk._register_changes()
test_eq(trk.count(), 1)
test_eq(trk.entry('file.jpg'), (_sha1(b'hello'), _mtime_s(f1), 1))

trk.set_entry('file.jpg', _sha1(b'hello'), _mtime_s(f1), False)     # as if it synced
os.utime(f1); _backdate(trk.folder)
trk._register_changes()
test_eq(trk.entry('file.jpg')[2], 0)                                # touched, content unchanged: still clean

f1.write_bytes(b'hello1'); _backdate(f1); _backdate(trk.folder)
trk._register_changes()
test_eq(trk.entry('file.jpg'), (_sha1(b'hello1'), _mtime_s(f1), 1))

trk.set_entry('file.jpg', _sha1(b'hello1'), _mtime_s(f1), False)
(trk.folder/'Thumbs.db').write_bytes(b'x')                          # never syncs
(trk.folder/'empty.gif').touch()                                    # nor do empty files
f1.unlink(); _backdate(trk.folder)
trk._register_changes()
test_eq(trk.count(), 0)
test_eq(trk.entry('file.jpg'), (None, 0, 1))                        # a deletion tombstone, awaiting upload
trk.pending(25)

[('file.jpg', None, 0)]

## Zips on the wire

Media sync transfers uncompressed zip archives. Upload batches contain at most 25 entries. A batch stops adding files once it exceeds 2.5MiB.

Zip entries use numeric names. `_meta` maps those names to filenames:

- Uploads use `[filename, zip_name]` pairs. A null `zip_name` records a deletion.
- Downloads use a `{zip_name: filename}` mapping. The server reports deletions in the change list, not in the zip.

In [ ]:
MAX_ZIP_FILES = 25
TARGET_ZIP_BYTES = int(2.5*1024*1024)

def _zip_up(entries):
    "Zip [(fname, data|None)] for upload; None data records a deletion"
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_STORED) as z:
        meta = []
        for i,(fname,data) in enumerate(entries):
            if data is None: meta.append((fname, None))
            else:
                z.writestr(str(i), data)
                meta.append((fname, str(i)))
        z.writestr('_meta', json.dumps(meta))
    return buf.getvalue()

def _unzip_down(zdata):
    "[(fname, data)] from a downloaded zip"
    with zipfile.ZipFile(io.BytesIO(zdata)) as z:
        names = json.loads(z.read('_meta'))
        return [(names[i.filename], z.read(i.filename)) for i in z.infolist() if i.filename!='_meta']

@patch
def _gather_zip(self:Media, entries):
    "[(fname, data|None)] to upload for pending `entries`; None if bad entries were pruned and the batch must be rebuilt"
    bad,out,size = [],[],0
    for fname,csum,_ in entries:
        if size>TARGET_ZIP_BYTES: break
        if sys.platform=='darwin' and unicodedata.normalize('NFC', fname)!=fname:   # pre-normalization Anki left NFD names in dbs
            bad.append(fname)
            continue
        data = None
        if csum is not None and (self.folder/fname).exists():
            data = (self.folder/fname).read_bytes()
            if not data or len(data)>MAX_FILE_SIZE or norm_fname(fname)!=fname:
                bad.append(fname)
                continue
            size += len(data)
        out.append((fname, data))
    if not bad: return out
    for f in bad: self.con.execute('delete from media where fname=?', (f,))
    return None

In [ ]:
up = _zip_up([('a.jpg', b'aaa'), ('gone.mp3', None), ('b.png', b'bb')])
with zipfile.ZipFile(io.BytesIO(up)) as z:
    test_eq(json.loads(z.read('_meta')), [['a.jpg', '0'], ['gone.mp3', None], ['b.png', '2']])
    test_eq(z.read('0'), b'aaa')

down = io.BytesIO()
with zipfile.ZipFile(down, 'w', zipfile.ZIP_STORED) as z:      # what a server download looks like
    z.writestr('0', b'data0'); z.writestr('_meta', json.dumps({'0':'pic.jpg'}))
got = _unzip_down(down.getvalue())
test_eq(got, [('pic.jpg', b'data0')])
got

[('pic.jpg', b'data0')]

Files can change after the scan. `_gather_zip` reads their current contents and treats missing files as deletions. It removes entries that no longer meet the upload rules. The caller then rebuilds the batch:

In [ ]:
gz = Media(td/'gz'/'media', td/'gz'/'collection.mdb')
for i in range(3): (gz.folder/f'big{i}.bin').write_bytes(bytes(1500_000))
(gz.folder/'small.txt').write_bytes(b'hi')
gz._register_changes()

got = gz._gather_zip([(f'big{i}.bin', _sha1(bytes(1500_000)), 0) for i in range(3)])
test_eq([f for f,_ in got], ['big0.bin', 'big1.bin'])               # 2.5MB cutoff: third file waits for the next batch

got = gz._gather_zip([('small.txt', _sha1(b'hi'), 0), ('vanished.png', 'deadbeef', 0), ('del.png', None, 0)])
test_eq(got, [('small.txt', b'hi'), ('vanished.png', None), ('del.png', None)])   # missing file -> uploaded as deletion

gz.set_entry('empty.bin', _sha1(b''), 1, True)                      # a zero-byte file can never upload
(gz.folder/'empty.bin').touch()
test_is(gz._gather_zip([('empty.bin', _sha1(b''), 1)]), None)       # pruned; batch must be rebuilt...
test_eq([f for f,_,_ in gz.pending(25)], ['big0.bin', 'big1.bin', 'big2.bin', 'small.txt'])   # ...without it

## Talking to the server

`SyncServer` uses `prefix='msync'` for media requests instead of collection sync's `sync/`. Both use a zstd body and the `anki-sync` header.

Media JSON replies have the form `{"data": ..., "err": ""}`. Change-list rows are `[fname, usn, sha1]` arrays. Upload replies are `[processed, current_usn]` arrays.

The media methods are `begin`, `mediaChanges`, `downloadFiles`, `uploadChanges` and `mediaSanity`. `begin` returns the server's current media USN.

In [ ]:
class MediaSanityFailed(Exception): pass

@patch
def _mjson(self:SyncServer, method, obj=None, data=None):
    "POST to msync/`method`, unwrapping the legacy {data,err} envelope of media replies"
    r = json.loads(self(method, obj, data, prefix='msync'))
    if r.get('err'): raise Exception(f"Media sync: {r['err']}")
    return r['data']

## Deciding what to do with a server change

Each server change gives a filename and its current SHA-1 checksum. An empty checksum means the server deleted the file. `required_change` compares this with the local checksum and upload status:

- Download the server's file if it's missing locally or has different content.
- Keep a local file awaiting upload when the server reports a deletion.
- Apply other server deletions locally.
- Clear the upload flag when both files have the same content.

The tests cover Anki's nine cases from `changes.rs`.

In [ ]:
def required_change(local, remote, state):
    "Resolve one server row: sha1s ('' means deleted), `state` in none/clean/dirty -> None, 'download', 'delete', or 'clean'"
    if not local and not remote: return None if state=='none' else 'delete'
    if not remote: return None if state=='dirty' else 'delete'
    if not local: return 'download'
    if local==remote: return 'clean' if state=='dirty' else None
    return 'download'

@patch
def _required_changes(self:Media, batch):
    "Sort server rows into filenames (to_download, to_delete, to_mark_clean)"
    dl,rm,cl = [],[],[]
    for fname,usn,rsum in batch:
        e = self.entry(fname)
        local,state = (e[0] or '', 'dirty' if e[2] else 'clean') if e else ('', 'none')
        c = required_change(local, rsum, state)
        if c=='download': dl.append(fname)
        elif c=='delete': rm.append(fname)
        elif c=='clean': cl.append(fname)
    return dl,rm,cl

In [ ]:
for args,exp in [(('','','none'),None),      (('','','clean'),'delete'),   (('','1','dirty'),'download'),
                 (('1','','dirty'),None),    (('1','','clean'),'delete'),  (('1','1','clean'),None),
                 (('1','1','dirty'),'clean'),(('a','b','dirty'),'download'),(('a','b','clean'),'download')]:
    test_eq(required_change(*args), exp)

## Files in and out

Remote deletions move files to a sibling `media.trash` folder. You can restore them manually.

Some files on AnkiWeb have names from older clients that no longer meet the filename rules. `_add_from_server` saves them under corrected names. It records the old name as a pending deletion and the corrected file as a pending upload.

In [ ]:
@patch
def _trash_files(self:Media, fnames):
    "Move `fnames` from the media folder into a sibling media.trash folder"
    if not fnames: return
    trash = self.folder.with_name('media.trash')
    trash.mkdir(exist_ok=True)
    for f in fnames:
        p = self.folder/f
        if p.exists(): p.replace(trash/f)

@patch
def _add_from_server(self:Media, fname, data):
    "Write a downloaded file; returns the db rows to record, as (fname, csum, mtime, dirty) tuples"
    if norm_fname(fname)==fname:
        p = self.folder/fname
        p.write_bytes(data)
        return [(fname, _sha1(data), _mtime_s(p), False)]
    used = add_media_file(self.folder, data, fname)                    # a pre-normalization name: store it corrected
    return [(fname, None, 0, True), (used, _sha1(data), _mtime_s(self.folder/used), True)]

In [ ]:
tf = Media(td/'tf'/'media', td/'tf'/'collection.mdb')
test_eq(tf._add_from_server('ok.png', b'fine'), [('ok.png', _sha1(b'fine'), _mtime_s(tf.folder/'ok.png'), False)])
bad = 'we|ird.png'                                                   # a name today's Anki would never create
test_eq(tf._add_from_server(bad, b'x'), [(bad, None, 0, True), ('weird.png', _sha1(b'x'), _mtime_s(tf.folder/'weird.png'), True)])
tf._trash_files(['ok.png', 'never-existed.png'])
test_eq((tf.folder/'ok.png').exists(), False)
test_eq((tf.folder.with_name('media.trash')/'ok.png').read_bytes(), b'fine')

## The sync procedure

`sync` scans local files, downloads server changes, then uploads pending local changes. It gets the server's media USN from `begin` unless the caller provides it from collection sync's `meta`. After exchanging changes, it asks the server to compare file counts. A sync with no changes skips this check.

Each downloaded batch updates the database and `lastUsn` in one transaction. An interrupted sync resumes from the last recorded batch.

An upload advances `lastUsn` only when the server's new USN equals the old `lastUsn` plus the number of entries it accepted. Otherwise, another client has uploaded concurrently. Keeping `lastUsn` unchanged lets the next sync fetch that client's changes.

A failed count check clears local sync state and raises `MediaSanityFailed`. It leaves the files intact. The next sync compares all files with the server again.

In [ ]:
@patch
def sync(self:Media, srv, server_usn=None):
    "Sync the media folder with `srv`: pull server changes, push local ones, then verify counts"
    self._register_changes()
    if server_usn is None: server_usn = srv._mjson('begin', dict(v=CLIENT_VER))['usn']
    acted = False
    if self.meta()[1]!=server_usn:
        self._fetch_changes(srv)
        acted = True
    if self.pending(1):
        self._send_changes(srv)
        acted = True
    if acted: self._finalize(srv)

@patch
def _fetch_changes(self:Media, srv):
    while True:
        batch = srv._mjson('mediaChanges', dict(lastUsn=self.meta()[1]))
        if not batch: return
        dl,rm,cl = self._required_changes(batch)
        self._trash_files(rm)
        added = []
        for chunk in chunked(dl, MAX_ZIP_FILES):
            zdata = srv('downloadFiles', dict(files=chunk), prefix='msync')
            for fname,data in _unzip_down(zdata): added += self._add_from_server(fname, data)
        self.con.execute('BEGIN IMMEDIATE')
        try:
            for f in cl: self.con.execute('update media set dirty=0 where fname=?', (f,))
            for f in rm: self.con.execute('delete from media where fname=?', (f,))
            for e in added: self.set_entry(*e)
            self.set_meta(_mtime_ms(self.folder), batch[-1][1])
            self.con.execute('COMMIT')
        except: self.con.execute('ROLLBACK'); raise

@patch
def _send_changes(self:Media, srv):
    while True:
        entries = self.pending(MAX_ZIP_FILES)
        if not entries: return
        batch = self._gather_zip(entries)
        if batch is None: continue                       # unuploadable entries were pruned; rebuild the batch
        processed,current_usn = srv._mjson('uploadChanges', data=_zip_up(batch))
        self.con.execute('BEGIN IMMEDIATE')
        try:
            done = [f for f,_ in batch[:processed]]
            for f in done: self.con.execute('update media set dirty=0 where fname=?', (f,))
            dirmod,last_usn = self.meta()
            # adopt the server's usn only if we produced it all; otherwise another client also uploaded, and the next fetch must see their changes
            if last_usn+len(done)==current_usn: self.set_meta(dirmod, current_usn)
            self.con.execute('COMMIT')
        except: self.con.execute('ROLLBACK'); raise

@patch
def _finalize(self:Media, srv):
    if srv._mjson('mediaSanity', dict(local=self.count()))!='OK':
        self.force_resync()
        raise MediaSanityFailed()

## Media on a Collection

Use `Collection.add_media` to copy a file into the media folder. It returns the filename to use in a note field: `<img src="name">` for pictures or `[sound:name]` for audio.

`Collection.media()` opens the media folder and database as a context manager. `Collection.sync_media()` syncs using the saved authentication.

In [ ]:
@patch
def media(self:Collection):
    "This collection's `Media`; close it after use (or use as a context manager)"
    return Media(self.path.with_suffix('.media'), self.path.with_suffix('.mdb'))

@patch
def add_media(self:Collection, file, fname=None):
    "Copy `file` (a path, or bytes with `fname`) into the media folder; returns the name to reference in note fields"
    if fname is None: fname = Path(file).name
    data = file if isinstance(file, bytes) else Path(file).read_bytes()
    return add_media_file(self.path.with_suffix('.media'), data, fname)

@patch
def sync_media(self:Collection, srv=None):
    "Media-sync with `srv` (default: the saved auth from a previous `sync`)"
    if srv is None: srv = self.load_auth()
    if srv is None: raise ValueError('No saved auth; sync() first')
    with self.media() as m: m.sync(srv)

In [ ]:
col = Collection.open(td/'ex'/'collection.anki2')
name = col.add_media(b'not really a png', 'diagram.png')
n = col.add(Front=f'What does this show? <img src="{name}">', Back='the architecture')
test_eq(name, 'diagram.png')
test_eq((td/'ex'/'collection.media'/name).read_bytes(), b'not really a png')
with col.media() as cm:
    cm._register_changes()
    test_eq(cm.pending(25), [(name, _sha1(b'not really a png'), _mtime_s(td/'ex'/'collection.media'/name))])
col.close()
n

<div class="prose" markdown="1">

**Front**: What does this show? <img src="diagram.png"> | **Back**: the architecture | 🏷 -

</div>

## Against a real server

These examples use the `anki` wheel's sync server on localhost, as in the syncer notebook. Anki is the second client. Its media sync runs in a backend thread. Poll `media_sync_status` until its `active` flag is false. The call raises an exception if sync fails.

The examples require a server process and have `eval: false`. `tests/test_sync.py` runs this sequence under pytest. It also covers batching and count-check failures.

In [ ]:
#| eval: false
import time


In [ ]:
#| eval: false
server, EP = start_sync_server(td/'server')

In [ ]:
#| eval: false
mcol = Collection.open(td/'sync'/'collection.anki2')
img = mcol.add_media(b'PNG our image', 'pic.png')
mcol.add(Front=f'What is this? <img src="{img}">', Back='our pic')
mcol.sync(user='tester', passw='s3kret', endpoint=EP, upload=True)
mcol.sync_media()
mcol.sync_media()      # caught up: a no-op

Check that Anki downloads the image and resolves its reference:

In [ ]:
#| eval: false
(td/'sync2').mkdir()
moc = AnkiCollection(str(td/'sync2'/'collection.anki2'))
auth = moc.sync_login('tester', 's3kret', endpoint=EP)
st = moc.sync_collection(auth, sync_media=False)
moc.close_for_full_sync()
moc.full_upload_or_download(auth=auth, server_usn=st.server_media_usn, upload=False)
moc.reopen(after_full_sync=True)
moc.sync_media(auth)
while moc.media_sync_status().active: time.sleep(0.05)
test_eq(Path(moc.media.dir(), 'pic.png').read_bytes(), b'PNG our image')
chk = moc.media.check()
test_eq((list(chk.missing), list(chk.unused)), ([], []))

Add an audio file and delete the picture in Anki. Sync Anki, then fastanki. Check that fastanki downloads the audio, moves the picture to `media.trash`, and has no pending uploads:

In [ ]:
#| eval: false
(td/'oracle.mp3').write_bytes(b'ORACLE AUDIO')
oname = moc.media.add_file(str(td/'oracle.mp3'))
moc.media.trash_files(['pic.png'])
moc.sync_media(auth)
while moc.media_sync_status().active: time.sleep(0.05)

mcol.sync_media()
with mcol.media() as m:
    test_eq((m.folder/oname).read_bytes(), b'ORACLE AUDIO')
    test_eq((m.folder/'pic.png').exists(), False)
    test_eq((m.folder.with_name('media.trash')/'pic.png').exists(), True)
    test_eq((m.pending(25), m.count()), ([], 1))
moc.close()
mcol.close()
server.terminate()